# Youtube Analysis for Hantavirus

## Introduction

## How to Run the Notebook

## 1. YouTube API Client
This function creates a connection to the YouTube Data API using an API key, allowing the program to retrieve video and comment data.

The data is collected using the YouTube Data API v3.  
A set of highly relevant videos was selected using keyword-based search.  
Comments were collected from these videos for later sentiment and text analysis.

In [1]:
!pip install google-api-python-client

In [ ]:
# from example given youtubeClient.py
# @author Hexu Chen, RMIT University, 2026
# @author Chenglong Ma, RMIT University, 2026

from googleapiclient.discovery import build

def youtubeClient():
    apiKey = "apikey" # this was my api key
    youtube = build('youtube', 'v3', developerKey=apiKey)
    return youtube

## 2. Fetching Youtube Data
This section collects Hantavirus-related YouTube videos and comments using the YouTube Data API. Multiple search queries are used to capture news, public health, expert explanation and general outbreak discussion videos.

This section does not need to be re-run once the data has been saved. The collected dataset is saved as:

`Data/hantavirus_youtube_data.json`

In [3]:
import json
import os

In [4]:
SEARCH_QUERIES = [
    "hantavirus outbreak news",
    "hantavirus cruise ship outbreak",
    "hantavirus cruise ship news",
    "hantavirus update",
    "hantavirus risk explained",
    "hantavirus symptoms transmission",
    "hantavirus expert explains",
    "hantavirus public health",
    "hantavirus 2026"
]

MAX_VIDEOS_PER_QUERY = 12
MAX_COMMENTS_PER_VIDEO = None
OUTPUT_FILE = "Data/hantavirus_youtube_data.json"

In [5]:
def fetchYoutubeDataMultipleQueries(
    searchQueries,
    maxVideosPerQuery=12,
    maxCommentsPerVideo=None,
    outputFile="Data/hantavirus_youtube_data.json"
):
    """
    Fetch YouTube videos and comments for multiple search queries.

    The function:
    - searches videos for each query
    - removes duplicate videos across search queries using videoId
    - fetches video statistics
    - fetches all available top-level comments if maxCommentsPerVideo is None
    - saves the final dataset to a JSON file
    """

    client = youtubeClient()

    os.makedirs(os.path.dirname(outputFile), exist_ok=True)

    videoIds = []
    videoSnippets = {}
    videoSearchQueries = {}

    # Step 1: Search for videos across multiple queries
    for query in searchQueries:
        print(f"Searching for videos with query: '{query}'...")

        searchResponse = client.search().list(
            q=query,
            part="snippet",
            type="video",
            order="relevance",
            maxResults=min(maxVideosPerQuery, 50)
        ).execute()

        for item in searchResponse.get("items", []):
            videoId = item["id"]["videoId"]

            # Avoid duplicate videos across search queries
            if videoId not in videoSnippets:
                videoIds.append(videoId)
                videoSnippets[videoId] = item["snippet"]
                videoSearchQueries[videoId] = query

    print(f"\nTotal unique videos found: {len(videoIds)}")

    if len(videoIds) == 0:
        print("No videos found. Check API setup or search queries.")
        return

    # Step 2: Fetch video statistics
    print("\nFetching video statistics...")

    videoStats = {}

    # videos.list can handle up to 50 video IDs per request
    for i in range(0, len(videoIds), 50):
        batchIds = videoIds[i:i + 50]

        statsResponse = client.videos().list(
            id=",".join(batchIds),
            part="statistics"
        ).execute()

        for item in statsResponse.get("items", []):
            videoStats[item["id"]] = item.get("statistics", {})

    # Step 3: Fetch comments
    print("\nFetching comments...")

    videos = []

    for videoId in videoIds:
        snippet = videoSnippets[videoId]
        stats = videoStats.get(videoId, {})

        video = {
            "videoId": videoId,
            "title": snippet.get("title", ""),
            "description": snippet.get("description", ""),
            "channelTitle": snippet.get("channelTitle", ""),
            "channelId": snippet.get("channelId", ""),
            "publishedAt": snippet.get("publishedAt", ""),
            "searchQuery": videoSearchQueries.get(videoId, ""),
            "url": f"https://www.youtube.com/watch?v={videoId}",
            "viewCount": int(stats.get("viewCount", 0)),
            "likeCount": int(stats.get("likeCount", 0)),
            "commentCount": int(stats.get("commentCount", 0)),
            "comments": []
        }

        try:
            commentsFetched = 0
            nextPageToken = None

            while True:
                # If maxCommentsPerVideo is None, fetch all comments page by page
                if maxCommentsPerVideo is not None:
                    requestLimit = min(100, maxCommentsPerVideo - commentsFetched)

                    if requestLimit <= 0:
                        break
                else:
                    requestLimit = 100

                commentResponse = client.commentThreads().list(
                    videoId=videoId,
                    part="snippet",
                    maxResults=requestLimit,
                    pageToken=nextPageToken,
                    textFormat="plainText",
                    order="relevance"
                ).execute()

                for commentThread in commentResponse.get("items", []):
                    topComment = commentThread["snippet"]["topLevelComment"]["snippet"]

                    video["comments"].append({
                        "commentId": commentThread.get("id", ""),
                        "author": topComment.get("authorDisplayName", ""),
                        "text": topComment.get("textDisplay", ""),
                        "publishedAt": topComment.get("publishedAt", ""),
                        "likeCount": topComment.get("likeCount", 0)
                    })

                    commentsFetched += 1

                    if maxCommentsPerVideo is not None and commentsFetched >= maxCommentsPerVideo:
                        break

                if maxCommentsPerVideo is not None and commentsFetched >= maxCommentsPerVideo:
                    break

                nextPageToken = commentResponse.get("nextPageToken")

                if not nextPageToken:
                    break

            print(f"{video['title'][:70]}... -> {len(video['comments'])} comments")

        except Exception as e:
            print(f"{video['title'][:70]}... -> Comments disabled or error: {e}")

        videos.append(video)

    # Step 4: Save to JSON
    data = {
        "searchQueries": searchQueries,
        "maxVideosPerQuery": maxVideosPerQuery,
        "maxCommentsPerVideo": maxCommentsPerVideo,
        "videos": videos
    }

    with open(outputFile, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"\nDone. Saved {len(videos)} unique videos to '{outputFile}'.")

In [6]:
fetchYoutubeDataMultipleQueries(
    SEARCH_QUERIES,
    maxVideosPerQuery=MAX_VIDEOS_PER_QUERY,
    maxCommentsPerVideo=MAX_COMMENTS_PER_VIDEO,
    outputFile=OUTPUT_FILE
)

Searching for videos with query: 'hantavirus outbreak news'...
Searching for videos with query: 'hantavirus cruise ship outbreak'...
Searching for videos with query: 'hantavirus cruise ship news'...
Searching for videos with query: 'hantavirus update'...
Searching for videos with query: 'hantavirus risk explained'...
Searching for videos with query: 'hantavirus symptoms transmission'...
Searching for videos with query: 'hantavirus expert explains'...
Searching for videos with query: 'hantavirus public health'...
Searching for videos with query: 'hantavirus 2026'...

Total unique videos found: 86

Fetching video statistics...

Fetching comments...
Hantavirus cruise ship docks after seven weeks at sea... -> 2 comments
Hantavirus outbreak sparks global health concerns and panic-driven spe... -> 17 comments
Cruise ship linked to deadly hantavirus outbreak to reach Netherlands ... -> 5 comments
Hantavirus Outbreak | Learn English with the News... -> 25 comments
Last people aboard hantavirus

## Install Required Packages (if needed)
If runnning the notebook for the firstime, install the required libraries using:

In [7]:
!pip install pandas matplotlib nltk numpy scikit-learn langid wordcloud

## 3. Load Dataset

In [8]:
import json

files = "Data/hantavirus_youtube_data.json"

with open(files, "r", encoding="utf-8") as f:
    data = json.load(f)

all_videos = data["videos"]

print("Total videos:", len(all_videos))
print("Search queries used:", len(data["searchQueries"]))

Total videos: 86
Search queries used: 9


In [9]:
comment_counts = [len(video["comments"]) for video in all_videos]

total_comments = sum(comment_counts)
min_comments = min(comment_counts)
max_comments = max(comment_counts)
avg_comments = sum(comment_counts) / len(comment_counts)

print("Total videos:", len(all_videos))
print("Total comments:", total_comments)
print("Minimum comments per video:", min_comments)
print("Maximum comments per video:", max_comments)
print("Average comments per video:", round(avg_comments, 2))

print("\nComments fetched per video:")
for video in all_videos:
    print(f"{video['title'][:70]}... -> {len(video['comments'])} comments")

Total videos: 86
Total comments: 12042
Minimum comments per video: 0
Maximum comments per video: 1256
Average comments per video: 140.02

Comments fetched per video:
Hantavirus cruise ship docks after seven weeks at sea... -> 2 comments
Hantavirus outbreak sparks global health concerns and panic-driven spe... -> 17 comments
Cruise ship linked to deadly hantavirus outbreak to reach Netherlands ... -> 5 comments
Hantavirus Outbreak | Learn English with the News... -> 25 comments
Last people aboard hantavirus ship finally disembark... -> 1 comments
What are the origins of hantavirus? How the virus tied to the cruise s... -> 15 comments
Cruise ship hit by deadly hantavirus outbreak arrives in the Netherlan... -> 0 comments
🚢 🤢 Hantavirus Cruise Ship Outbreak: “Close Contact” - What It Really ... -> 70 comments
How Worried Should We Be About Hantavirus?... -> 43 comments
LIVE: Hantavirus Outbreak News: 3 Dead On Cruise Ship | WHO Chief Brie... -> 3 comments
Officials tracking at least 41 Am

In [10]:
# convert to df
import pandas as pd

video_rows = []
comment_rows = []

for video in all_videos:
    video_rows.append({
        "videoId": video["videoId"],
        "title": video["title"],
        "description": video["description"],
        "channelTitle": video["channelTitle"],
        "channelId": video["channelId"],
        "publishedAt": video["publishedAt"],
        "searchQuery": video["searchQuery"],
        "url": video["url"],
        "viewCount": video["viewCount"],
        "likeCount": video["likeCount"],
        "commentCount": video["commentCount"],
        "commentsFetched": len(video["comments"])
    })

    for comment in video["comments"]:
        comment_rows.append({
            "commentId": comment["commentId"],
            "videoId": video["videoId"],
            "title": video["title"],
            "channelTitle": video["channelTitle"],
            "searchQuery": video["searchQuery"],
            "commentAuthor": comment["author"],
            "commentText": comment["text"],
            "commentPublishedAt": comment["publishedAt"],
            "commentLikeCount": comment["likeCount"]
        })

videos_df = pd.DataFrame(video_rows)
comments_df = pd.DataFrame(comment_rows)

print("Videos dataframe:", videos_df.shape)
print("Comments dataframe:", comments_df.shape)

videos_df.head()

Videos dataframe: (86, 12)
Comments dataframe: (12042, 9)


,videoId,title,description,channelTitle,channelId,publishedAt,searchQuery,url,viewCount,likeCount,commentCount,commentsFetched
0,V07bI_iyMVA,Hantavirus cruise ship docks after seven weeks...,For more context and news coverage of the most...,NBC News,UCeY0bbntWzzVIaj2z3QigXg,2026-05-18T12:58:33Z,hantavirus outbreak news,https://www.youtube.com/watch?v=V07bI_iyMVA,1558,28,3,2
1,KpZN0OA0GNc,Hantavirus outbreak sparks global health conce...,The Hantavirus outbreak linked to the MV Hondi...,SABC News,UC8yH-uI81UUtEMDsowQyx1g,2026-05-17T16:21:21Z,hantavirus outbreak news,https://www.youtube.com/watch?v=KpZN0OA0GNc,8521,0,25,17
2,4URktweTzV4,Cruise ship linked to deadly hantavirus outbre...,The cruise ship linked to the deadly hantaviru...,The New Indian Express,UCQtVJ-76KjPds9evTODkq-A,2026-05-18T08:08:09Z,hantavirus outbreak news,https://www.youtube.com/watch?v=4URktweTzV4,4046,24,6,5
3,COt1faxmdO0,Hantavirus Outbreak | Learn English with the News,Today you'll learn English with a news article...,JForrest English,UCLNBasuHKOILIwewAetJO6g,2026-05-18T12:00:30Z,hantavirus outbreak news,https://www.youtube.com/watch?v=COt1faxmdO0,1587,275,28,25
4,4KWbgoo60D4,Last people aboard hantavirus ship finally dis...,The MV Hondius was carrying 25 crew members an...,Sky News,UCoMdktPbSTixAyNGwb-UYkQ,2026-05-18T12:01:14Z,hantavirus outbreak news,https://www.youtube.com/watch?v=4KWbgoo60D4,1643,17,1,1


In [11]:
videos_df.to_csv("Data/youtube_videos.csv", index=False)
comments_df.to_csv("Data/youtube_comments.csv", index=False)

print("Saved youtube_videos.csv and youtube_comments.csv")

Saved youtube_videos.csv and youtube_comments.csv
